In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableSequence

# Функция, выступающая как LLM (заглушка)
def my_llm(prompt: str) -> str:
    return f"Ответ модели на: {prompt}"

# Создаем runnable с функцией LLM
llm_runnable = RunnableLambda(my_llm)

# Дополнительная обработка вывода: переворот строки
process_output = RunnableLambda(lambda text: text[::-1])

# Создаем цепочку из вызова LLM и обработки результата
chain = llm_runnable | process_output

# Запускаем цепочку
result = chain.invoke("Привет, LangChain!")
print(result)  # Выведет перевернутый ответ модели

!niahCgnaL ,тевирП :ан иледом тевтО


In [8]:
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

# Создание шаблона с историей диалога
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты ассистент, говорящий по-русски."),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

# Подстановка истории и вопроса
prompt_value = prompt.invoke({
    "history": [
        {"role": "human", "content": "Привет! Как дела?"},
        {"role": "ai", "content": "Все отлично!"}
    ],
    "question": "Какая погода в Москве?"
})

# Передача подготовленных сообщений в LLM
def llm_call(messages):
    # Здесь вызов реальной LLM, например OpenAI
    return f"Ответ на: {messages[-1].content}"

llm_runnable = RunnableLambda(llm_call)
response = llm_runnable.invoke(prompt_value.to_messages())
print(response)


Ответ на: Какая погода в Москве?


In [9]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)

# Несколько примеров (вопрос-ответ)
examples = [
    {"input": "2+2", "output": "4"},
    {"input": "2+3", "output": "5"},
]

# Шаблон для одного примера — превращает словарь в пару сообщений
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),  # вопрос
    ("ai", "{output}")     # ответ
])

# Пример few-shot шаблона, объединяющего эти примеры
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Итоговый шаблон с системным сообщением, примерами и новым вопросом
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an excellent math solver."),
    few_shot_prompt,
    ("human", "{input}")
])

# Формируем сообщения для нового вопроса
messages = final_prompt.invoke({"input": "What is 2+5?"}).to_messages()

for msg in messages:
    print(f"{msg.type}: {msg.content}")


system: You are an excellent math solver.
human: 2+2
ai: 4
human: 2+3
ai: 5
human: What is 2+5?


In [4]:
from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms.base import LLM
from typing import Optional, List, Any

In [5]:
class MyLLM(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        return f"Ответ модели на: {prompt}"

    @property
    def _identifying_params(self) -> dict:
        return {}

    @property
    def _llm_type(self) -> str:
        return "my_custom_llm"

In [7]:
# Шаблон промпта для помощника по учёбе
study_helper_template = """
I want you to act as a study helper for a student.
Provide a list of study tips and recommend resources that would be beneficial for {study_topic}.
Ensure that the tips are practical and the resources are reliable and relevant.
What are some effective study strategies and resources for {study_topic}?
"""

# Создание объекта PromptTemplate с необходимыми переменными
prompt_template = PromptTemplate(
    input_variables=["study_topic"],
    template=study_helper_template,
)

# Описание учебной задачи или предмета
study_description = "preparing for a final exam in advanced calculus"

# Форматирование промпта с использованием описания
prompt_template.format(study_topic=study_description)

# Создаем экземпляр кастомного LLM
llm = MyLLM()

chain = LLMChain(llm=llm, prompt=prompt_template)

# Запуск цепочки и вывод результата
response = chain.run(study_description)
print(response)

Ответ модели на: 
I want you to act as a study helper for a student.
Provide a list of study tips and recommend resources that would be beneficial for preparing for a final exam in advanced calculus.
Ensure that the tips are practical and the resources are reliable and relevant.
What are some effective study strategies and resources for preparing for a final exam in advanced calculus?

